# Практическая работа №3. Распределённая модель обработки данных

**Часть 2. Самостоятельная работа: распределённый анализ продаж КОМУС**

## 1. Постановка задачи и источник данных

Использован набор транзакционных данных компании КОМУС, предоставленный в составе материалов практической работы и применяемый в учебных целях. Набор состоит из четырёх CSV-файлов:

| Файл | Строк | Содержание |
|---|---|---|
| `Продажи.csv` | 865 222 | Таблица фактов: строки продаж с себестоимостью, суммой, скидкой, количеством, филиалом и доставкой |
| `Продукты.csv` | 21 163 | Справочник товаров: название, бренд, товарная группа |
| `Клиенты.csv` | 3 838 | Справочник клиентов — юридических лиц |
| `Суммы чеков.csv` | 160 114 | Итоговые суммы чеков; в анализе не используется |

Клиенты представлены юридическими лицами и одной обезличенной записью «Частное лицо», объединяющей розничных покупателей; персональные данные физических лиц в наборе отсутствуют.

**Цель анализа** — средствами Spark сформировать аналитическую витрину продаж и оценить выручку, валовую маржу и доставку в разрезе филиалов, месяцев, товарных групп и клиентов.

Файлы из архива `Транзакции КОМУС.zip` должны находиться в папке `komus` рядом с ноутбуком.

## 2. SparkSession и версии окружения

In [1]:
import os
import sys
import time
import subprocess
from pathlib import Path

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Practice3_Komus")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

java = subprocess.run(["java", "-version"], capture_output=True, text=True)

print("Python:", sys.version.split()[0])
print("Java:", java.stderr.splitlines()[0])
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Ядер (defaultParallelism):", spark.sparkContext.defaultParallelism)
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.driver.memory:", spark.conf.get("spark.driver.memory"))

Python: 3.11.16
Java: java version "21.0.9" 2025-10-21 LTS
Spark: 4.2.0
Master: local[*]
Ядер (defaultParallelism): 20
spark.sql.shuffle.partitions: 8
spark.driver.memory: 2g


**Вывод.** Создана локальная сессия Spark в режиме `local[*]`; версии Python, Java и Spark зафиксированы выше. Число разделов после перераспределяющих операций уменьшено до 8, объём памяти драйвера — 2 ГБ, что достаточно для таблицы фактов объёмом около 90 МБ.

## 3. Подготовка источника и чтение основной таблицы

Исходные файлы уже имеют формат CSV, который Spark читает встроенными средствами, поэтому отдельный staging-этап не требуется. Достаточно явно описать формат файлов: разделитель — табуляция, кодировка UTF-8, первая строка содержит заголовки, кавычки внутри значений экранированы удвоением, а пропуски записаны текстом `NULL`.

In [2]:
KOMUS = Path("komus")

for f in sorted(KOMUS.glob("*.csv")):
    print(f"{f.name:20s} {f.stat().st_size / 1024 / 1024:7.1f} МБ")

CSV_OPTIONS = {
    "header": True,
    "sep": "\t",
    "encoding": "UTF-8",
    "nullValue": "NULL",
    "escape": '"',
}

sales_raw = spark.read.options(**CSV_OPTIONS).csv(str(KOMUS / "Продажи.csv"))

print("Строк:", sales_raw.count())
print("Столбцов:", len(sales_raw.columns))
print("Столбцы:", sales_raw.columns)
sales_raw.printSchema()

Клиенты.csv              0.5 МБ
Продажи.csv             89.0 МБ
Продукты.csv             3.3 МБ
Суммы чеков.csv         13.2 МБ
Строк: 865222
Столбцов: 11
Столбцы: ['Себестоимость', 'Дата покупки', 'Номер чека', 'Product_key', 'Филиал', 'Количество', 'Сумма покупки', 'Сумма скидки', 'Client_ID', 'Машина доставки', 'Адрес']
root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: string (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Адрес: string (nullable = true)



**Вывод.** Основная таблица «Продажи» прочитана напрямую из CSV без staging-этапа: 865 222 строки и 11 столбцов. Без указания схемы все столбцы получили тип `string`, а их имена записаны кириллицей с пробелами, что неудобно для выражений и SQL-запросов.

## 4. Явная схема StructType и сравнение с inferSchema

In [3]:
sales_schema = StructType([
    StructField("cost", StringType(), True),
    StructField("purchase_date", StringType(), True),
    StructField("receipt_no", StringType(), True),
    StructField("product_key", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("client_id", StringType(), True),
    StructField("vehicle", StringType(), True),
    StructField("address", StringType(), True),
])

start = time.time()
sales_typed = spark.read.options(**CSV_OPTIONS).schema(sales_schema).csv(str(KOMUS / "Продажи.csv"))
rows_typed = sales_typed.count()
time_typed = time.time() - start

start = time.time()
sales_inferred = (
    spark.read.options(**CSV_OPTIONS)
    .option("inferSchema", True)
    .csv(str(KOMUS / "Продажи.csv"))
)
rows_inferred = sales_inferred.count()
time_inferred = time.time() - start

print(f"Явная схема: {rows_typed} строк, {time_typed:.1f} с")
print(f"inferSchema: {rows_inferred} строк, {time_inferred:.1f} с")

sales_typed.printSchema()
sales_inferred.printSchema()

Явная схема: 865222 строк, 0.4 с
inferSchema: 865222 строк, 1.3 с
root
 |-- cost: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- vehicle: string (nullable = true)
 |-- address: string (nullable = true)

root
 |-- Себестоимость: string (nullable = true)
 |-- Дата покупки: string (nullable = true)
 |-- Номер чека: string (nullable = true)
 |-- Product_key: string (nullable = true)
 |-- Филиал: string (nullable = true)
 |-- Количество: integer (nullable = true)
 |-- Сумма покупки: string (nullable = true)
 |-- Сумма скидки: string (nullable = true)
 |-- Client_ID: string (nullable = true)
 |-- Машина доставки: string (nullable = true)
 |-- Адрес: string (nullable = true)



In [4]:
sales_typed.select("cost", "purchase_date", "quantity", "amount", "discount").show(5)

+------+-------------+--------+------+--------+
|  cost|purchase_date|quantity|amount|discount|
+------+-------------+--------+------+--------+
|113,33|   27.09.2020|      25|   250|       0|
|127,12|   27.09.2020|      25|   275|       0|
|498,19|   27.09.2020|      25|   875|       0|
|928,37|   27.09.2020|      25|  2125|       0|
|624,23|   27.09.2020|      25|   900|       0|
+------+-------------+--------+------+--------+
only showing top 5 rows


Денежные поля записаны с десятичной запятой (`113,33`), поэтому в схеме они оставлены строками. Проверка: что произойдёт, если объявить сумму покупки типом `DoubleType`.

In [5]:
wrong_schema = StructType([
    StructField("amount", DoubleType(), True) if f.name == "amount" else f
    for f in sales_schema.fields
])

wrong = spark.read.options(**CSV_OPTIONS).schema(wrong_schema).csv(str(KOMUS / "Продажи.csv"))

print("Пустых сумм при чтении строкой:", sales_typed.filter(F.col("amount").isNull()).count())
print("Пустых сумм при DoubleType:", wrong.filter(F.col("amount").isNull()).count())

Пустых сумм при чтении строкой: 0
Пустых сумм при DoubleType: 196690


**Вывод.** Явная схема `StructType` задала латинские имена столбцов и целочисленный тип количества; денежные поля и дата оставлены строками, поскольку записаны в русском формате (десятичная запятая, дата `дд.мм.гггг`). `inferSchema` распознал как число только количество, остальные поля оставил строками и при этом потребовал заметно больше времени из-за дополнительного прохода по данным. Объявление суммы покупки типом `DoubleType` привело бы к тому, что 196 690 значений с десятичной запятой без ошибки превратились бы в `null`, поэтому числовые типы задаются явным преобразованием после чтения.

## 5. Преобразования и вычисляемые признаки

In [6]:
def to_number(column):
    return F.regexp_replace(F.col(column), ",", ".").cast("double")


sales = (
    sales_typed
    .withColumn("cost", to_number("cost"))
    .withColumn("amount", to_number("amount"))
    .withColumn("discount", F.round(to_number("discount"), 2))
    .withColumn("purchase_date", F.to_date("purchase_date", "dd.MM.yyyy"))
    .withColumn("branch", F.trim(F.col("branch")))
    .withColumn("purchase_month", F.date_format("purchase_date", "yyyy-MM"))
    .withColumn("margin", F.round(F.col("amount") - F.col("cost"), 2))
    .withColumn("is_delivery", F.col("vehicle").isNotNull())
    .drop("address")
)

sales.printSchema()

print("Период:", sales.select(F.min("purchase_date"), F.max("purchase_date")).first())

root
 |-- cost: double (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- receipt_no: string (nullable = true)
 |-- product_key: string (nullable = true)
 |-- branch: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- client_id: string (nullable = true)
 |-- vehicle: string (nullable = true)
 |-- purchase_month: string (nullable = true)
 |-- margin: double (nullable = true)
 |-- is_delivery: boolean (nullable = false)

Период: Row(min(purchase_date)=datetime.date(2020, 6, 16), max(purchase_date)=datetime.date(2021, 12, 31))


In [7]:
print("Филиалы до trim:")
sales_typed.select("branch").distinct().orderBy("branch").show(truncate=False)

print("Филиалы после trim:")
sales.select("branch").distinct().orderBy("branch").show(truncate=False)

Филиалы до trim:
+-----------------------------+
|branch                       |
+-----------------------------+
| РС (Распределительный склад)|
| ТЗ-1 ( ул.Фадеева)          |
| ТЗ-2 ( Ленинградское шоссе) |
| ТЗ-3 (Курский)              |
|Багетная мастерская          |
|Кафе-1 ( ул.Фадеева)         |
+-----------------------------+

Филиалы после trim:
+----------------------------+
|branch                      |
+----------------------------+
|Багетная мастерская         |
|Кафе-1 ( ул.Фадеева)        |
|РС (Распределительный склад)|
|ТЗ-1 ( ул.Фадеева)          |
|ТЗ-2 ( Ленинградское шоссе) |
|ТЗ-3 (Курский)              |
+----------------------------+



In [8]:
large_lines = (
    sales
    .filter(F.col("amount") >= 10000)
    .select("purchase_date", "branch", "product_key", "quantity", "amount", "margin")
)

print("Строк продаж с суммой от 10 000:", large_lines.count())
large_lines.orderBy(F.desc("amount")).show(5, truncate=False)

print("Строк с нулевой суммой:", sales.filter(F.col("amount") == 0).count())
print("Строк со скидкой:", sales.filter(F.col("discount") > 0).count())

Строк продаж с суммой от 10 000: 284
+-------------+---------------------------+-----------+--------+--------+---------+
|purchase_date|branch                     |product_key|quantity|amount  |margin   |
+-------------+---------------------------+-----------+--------+--------+---------+
|2021-10-15   |ТЗ-2 ( Ленинградское шоссе)|6736_20_289|50      |231500.0|128740.94|
|2021-12-03   |ТЗ-2 ( Ленинградское шоссе)|8044_20_278|50      |122500.0|51073.02 |
|2021-09-12   |ТЗ-2 ( Ленинградское шоссе)|3265_73_297|125     |115750.0|61685.63 |
|2021-10-22   |ТЗ-2 ( Ленинградское шоссе)|4963_20_289|75      |110250.0|53029.03 |
|2021-07-10   |ТЗ-2 ( Ленинградское шоссе)|1160_77_46 |5       |93885.0 |50783.8  |
+-------------+---------------------------+-----------+--------+--------+---------+
only showing top 5 rows
Строк с нулевой суммой: 67
Строк со скидкой: 137757


**Вывод.** Выполнены преобразования `select()`, `filter()`, `withColumn()`, `drop()`, `distinct()` и `orderBy()`. Денежные поля приведены к `double` заменой запятой на точку, дата покупки — к типу `date`, из названий четырёх из шести филиалов удалены начальные пробелы. Созданы вычисляемые признаки: месяц покупки, валовая маржа строки (сумма покупки минус себестоимость) и признак доставки. Данные охватывают период с 16.06.2020 по 31.12.2021; продаж на сумму от 10 000 — 284, нулевую сумму имеют 67 строк, скидка предоставлена в 137 757 строках.

## 6. Пропуски и качество данных

In [9]:
check_cols = ["cost", "receipt_no", "quantity", "vehicle"]

print("Пропуски после чтения с nullValue='NULL':")
sales.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_cols
]).show(vertical=True)

no_null_option = (
    spark.read.option("header", True).option("sep", "\t")
    .csv(str(KOMUS / "Продажи.csv"))
    .toDF(*[f.name for f in sales_schema.fields])
)

print("Без nullValue: пустых значений и строк с текстом 'NULL':")
no_null_option.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c + "_null") for c in check_cols]
    + [F.sum((F.col(c) == "NULL").cast("int")).alias(c + "_text") for c in check_cols]
).show(vertical=True)

Пропуски после чтения с nullValue='NULL':
-RECORD 0------------
 cost       | 86431  
 receipt_no | 3934   
 quantity   | 100    
 vehicle    | 832181 

Без nullValue: пустых значений и строк с текстом 'NULL':
-RECORD 0-----------------
 cost_null       | 0      
 receipt_no_null | 0      
 quantity_null   | 0      
 vehicle_null    | 0      
 cost_text       | 86431  
 receipt_no_text | 3934   
 quantity_text   | 100    
 vehicle_text    | 832181 



In [10]:
(
    sales
    .groupBy("branch")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("cost").isNull().cast("int")).alias("no_cost"),
        F.sum(F.col("receipt_no").isNull().cast("int")).alias("no_receipt"),
        F.sum(F.col("quantity").isNull().cast("int")).alias("no_quantity"),
        F.sum(F.col("is_delivery").cast("int")).alias("delivery"),
    )
    .orderBy(F.desc("rows"))
    .show(truncate=False)
)

revenue_total = sales.agg(F.sum("amount")).first()[0]
revenue_no_cost = sales.filter(F.col("cost").isNull()).agg(F.sum("amount")).first()[0]
print(f"Доля выручки строк без себестоимости: {revenue_no_cost / revenue_total * 100:.1f}%")

+----------------------------+------+-------+----------+-----------+--------+
|branch                      |rows  |no_cost|no_receipt|no_quantity|delivery|
+----------------------------+------+-------+----------+-----------+--------+
|ТЗ-2 ( Ленинградское шоссе) |431372|42858  |2807      |43         |8379    |
|ТЗ-1 ( ул.Фадеева)          |238046|23799  |19        |33         |19201   |
|ТЗ-3 (Курский)              |190730|19243  |11        |24         |5212    |
|Кафе-1 ( ул.Фадеева)        |3977  |417    |0         |0          |0       |
|Багетная мастерская         |848   |91     |848       |0          |0       |
|РС (Распределительный склад)|249   |23     |249       |0          |249     |
+----------------------------+------+-------+----------+-----------+--------+

Доля выручки строк без себестоимости: 10.1%


In [11]:
sales_clean = sales.fillna({"vehicle": "без доставки"})

sales_clean.groupBy("vehicle").count().orderBy(F.desc("count")).show(truncate=False)

+------------+------+
|vehicle     |count |
+------------+------+
|без доставки|832181|
|Н290ЛО      |16510 |
|Б715ПН      |9020  |
|А645ДМ      |7511  |
+------------+------+



**Вывод.** Пропуски в файлах записаны текстом `NULL`: без опции `nullValue` Spark не обнаружил бы ни одного пропуска, а при приведении типов такие значения вызвали бы ошибку. Анализ по филиалам показал, что пропуски имеют разную природу, и решение принято для каждого поля отдельно:

- **машина доставки** отсутствует в 832 181 строке (96%) — это продажи без доставки, поэтому пропуск заполнен категорией «без доставки»;
- **номер чека** отсутствует в 3 934 строках: у всех продаж багетной мастерской и распределительного склада, а также у части заказов юрлиц; строки сохранены, так как отражают реальную выручку, а фиктивный номер исказил бы число чеков;
- **себестоимость** отсутствует в 86 431 строке (10%) равномерно во всех филиалах, на них приходится 10,1% выручки; строки сохранены для анализа выручки, а маржа по ним не рассчитывается;
- **количество** отсутствует в 100 строках; они сохранены, поскольку сумма покупки по ним известна.

Ни одна строка не удалена только из-за наличия `null`.

## 7. Агрегации

In [12]:
branch_stats = (
    sales_clean
    .groupBy("branch")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_no").alias("receipts"),
        F.round(F.sum("amount"), 2).cast("decimal(18,2)").alias("revenue"),
        F.round(F.sum("margin"), 2).cast("decimal(18,2)").alias("margin"),
        F.sum(F.when(F.col("cost").isNotNull(), F.col("amount"))).alias("revenue_with_cost"),
        F.round(F.avg("amount"), 2).alias("avg_line"),
        F.round(F.avg(F.col("is_delivery").cast("int")) * 100, 2).alias("delivery_pct"),
    )
    .withColumn(
        "margin_pct",
        F.when(
            F.col("revenue_with_cost") > 0,
            F.round(F.col("margin") / F.col("revenue_with_cost") * 100, 2),
        ),
    )
    .drop("revenue_with_cost")
    .orderBy(F.desc("revenue"))
)

branch_stats.show(truncate=False)

+----------------------------+------+--------+-----------+-----------+--------+------------+----------+
|branch                      |lines |receipts|revenue    |margin     |avg_line|delivery_pct|margin_pct|
+----------------------------+------+--------+-----------+-----------+--------+------------+----------+
|ТЗ-2 ( Ленинградское шоссе) |431372|78784   |58750803.24|26441601.20|136.2   |1.94        |50.06     |
|ТЗ-1 ( ул.Фадеева)          |238046|46337   |44343168.32|19917348.47|186.28  |8.07        |49.94     |
|ТЗ-3 (Курский)              |190730|32296   |26249055.61|11839145.49|137.62  |2.73        |50.21     |
|Багетная мастерская         |848   |0       |1399096.07 |636788.78  |1649.88 |0.0         |50.59     |
|РС (Распределительный склад)|249   |0       |788679.75  |354210.08  |3167.39 |100.0       |47.76     |
|Кафе-1 ( ул.Фадеева)        |3977  |2459    |207762.35  |93800.87   |52.24   |0.0         |50.47     |
+----------------------------+------+--------+-----------+------

In [13]:
monthly = (
    sales_clean
    .groupBy("purchase_month")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_no").alias("receipts"),
        F.round(F.sum("amount"), 2).cast("decimal(18,2)").alias("revenue"),
    )
    .orderBy("purchase_month")
)

monthly.show(30, truncate=False)

+--------------+------+--------+-----------+
|purchase_month|lines |receipts|revenue    |
+--------------+------+--------+-----------+
|2020-06       |2932  |662     |269717.58  |
|2020-07       |13218 |2438    |1317945.84 |
|2020-08       |11918 |2254    |1309760.80 |
|2020-09       |20421 |3646    |2121197.10 |
|2020-10       |21995 |4173    |2684682.71 |
|2020-11       |20728 |3982    |2245580.97 |
|2020-12       |25501 |4718    |3386536.22 |
|2021-01       |20506 |3653    |2562761.24 |
|2021-02       |25085 |4471    |2972448.33 |
|2021-03       |24446 |5030    |2829779.86 |
|2021-04       |21864 |4879    |2542105.50 |
|2021-05       |29985 |5982    |3534211.82 |
|2021-06       |32742 |6254    |4308661.66 |
|2021-07       |61749 |9451    |11193030.60|
|2021-08       |69165 |9094    |11631730.15|
|2021-09       |120121|16823   |19053262.60|
|2021-10       |107459|20566   |17447678.83|
|2021-11       |108614|19954   |17303145.01|
|2021-12       |126773|24013   |23024328.52|
+---------

In [14]:
(
    sales_clean
    .groupBy("branch")
    .agg(
        F.min("purchase_month").alias("first_month"),
        F.max("purchase_month").alias("last_month"),
        F.countDistinct("purchase_month").alias("months"),
    )
    .orderBy("first_month")
    .show(truncate=False)
)

+----------------------------+-----------+----------+------+
|branch                      |first_month|last_month|months|
+----------------------------+-----------+----------+------+
|ТЗ-2 ( Ленинградское шоссе) |2020-06    |2021-12   |19    |
|РС (Распределительный склад)|2020-09    |2021-01   |4     |
|ТЗ-1 ( ул.Фадеева)          |2021-01    |2021-12   |8     |
|Кафе-1 ( ул.Фадеева)        |2021-02    |2021-12   |11    |
|ТЗ-3 (Курский)              |2021-04    |2021-12   |9     |
|Багетная мастерская         |2021-07    |2021-12   |6     |
+----------------------------+-----------+----------+------+



**Вывод.** Наибольшую выручку даёт торговый зал ТЗ-2 на Ленинградском шоссе (58,75 млн, 431 372 строки), далее ТЗ-1 (44,34 млн) и ТЗ-3 (26,25 млн). Доля валовой маржи во всех филиалах близка к 50%. Доставка используется преимущественно в ТЗ-1 (8,07% строк) и распределительном складе (все продажи). Помесячная выручка резко растёт с июля 2021 года, однако это связано с подключением новых точек: ТЗ-3 присутствует в данных с апреля 2021 года, ТЗ-1 и багетная мастерская — с июля 2021 года (до этого у ТЗ-1 лишь три единичные продажи), а ранее продажи шли почти полностью через ТЗ-2.

## 8. Соединение со справочниками: LEFT JOIN и INNER JOIN

In [15]:
products = (
    spark.read.options(**CSV_OPTIONS)
    .csv(str(KOMUS / "Продукты.csv"))
    .toDF("product_key", "product_name", "brand", "product_group")
)

clients_raw = (
    spark.read.options(**CSV_OPTIONS)
    .csv(str(KOMUS / "Клиенты.csv"))
    .toDF("client_id", "buyer", "contractor_name")
)

print("Товаров:", products.count(), "| уникальных ключей:", products.select("product_key").distinct().count())
print("Строк клиентов:", clients_raw.count(), "| уникальных client_id:", clients_raw.select("client_id").distinct().count())

Товаров: 21163 | уникальных ключей: 21163
Строк клиентов: 3838 | уникальных client_id: 3227


In [16]:
sales_products = (
    sales_clean
    .join(products.select("product_key", "brand", "product_group"), on="product_key", how="left")
)

print("Строк до LEFT JOIN:", sales_clean.count())
print("Строк после LEFT JOIN:", sales_products.count())
print("Строк без найденного товара:", sales_products.filter(F.col("product_group").isNull()).count())
print("Строк без бренда:", sales_products.filter(F.col("brand").isNull()).count())

sales_products = sales_products.fillna({"brand": "unknown"})

Строк до LEFT JOIN: 865222
Строк после LEFT JOIN: 865222
Строк без найденного товара: 0
Строк без бренда: 27


In [17]:
naive = sales_products.join(clients_raw, on="client_id", how="inner")
print("INNER JOIN с исходным справочником клиентов:", naive.count())

clients = clients_raw.dropDuplicates(["client_id"]).select("client_id", "buyer")
mart = sales_products.join(clients, on="client_id", how="inner")

print("Строк до INNER JOIN:", sales_products.count())
print("Строк после INNER JOIN с уникальными клиентами:", mart.count())

INNER JOIN с исходным справочником клиентов: 874573
Строк до INNER JOIN: 865222
Строк после INNER JOIN с уникальными клиентами: 865222


**Вывод.** LEFT JOIN со справочником товаров сохранил все 865 222 строки: товар найден для каждой продажи, у 27 строк в справочнике не указан бренд, он заменён значением `unknown`. Справочник клиентов содержит 3 838 строк, но только 3 227 уникальных `client_id`, так как у клиента может быть несколько юрлиц-контрагентов. Поэтому INNER JOIN с исходным справочником увеличил число строк до 874 573, то есть размножил продажи. После удаления дубликатов по `client_id` INNER JOIN вернул ровно 865 222 строки. Тип INNER выбран потому, что продажа без клиента в справочнике была бы ошибкой данных, а совпадение числа строк до и после соединения подтверждает ссылочную целостность.

## 9. Spark SQL

In [18]:
mart.createOrReplaceTempView("komus_sales")

top_groups_sql = spark.sql('''
SELECT
    product_group,
    COUNT(*) AS lines,
    COUNT(DISTINCT receipt_no) AS receipts,
    ROUND(SUM(amount), 2) AS revenue,
    ROUND(SUM(margin), 2) AS margin
FROM komus_sales
GROUP BY product_group
ORDER BY revenue DESC
LIMIT 10
''')

top_groups_sql.show(truncate=False)

+-------------------------------------+-----+--------+----------+----------+
|product_group                        |lines|receipts|revenue   |margin    |
+-------------------------------------+-----+--------+----------+----------+
|Основы-холст на подрамнике-Базовый   |6703 |4656    |4407000.88|2022021.64|
|z-Уценка                             |37800|15342   |3296317.47|1478031.01|
|Живопись-акварель-наборы             |6582 |6047    |3182778.15|1447107.11|
|Бумага-Картон-пенокартон             |2531 |2199    |2716844.79|1213973.48|
|Подрамники, рамы-Планшеты            |3850 |3579    |2550475.59|1170236.11|
|Живопись-масло-профи-Мастер-Класс ЗХК|15925|5255    |2367607.02|1052157.01|
|Сумки-на молнии ткань/кожа           |2949 |2836    |2184469.42|983199.07 |
|Кисти-Масло-Синтетика                |24409|11811   |2103641.02|941731.68 |
|Графические материалы-маркеры-маркеры|15459|6921    |1948429.28|872331.34 |
|Мольберты-Этюдники                   |593  |587     |1855947.64|849012.66 |

In [19]:
top_groups_api = (
    mart
    .groupBy("product_group")
    .agg(
        F.count("*").alias("lines"),
        F.countDistinct("receipt_no").alias("receipts"),
        F.round(F.sum("amount"), 2).alias("revenue"),
        F.round(F.sum("margin"), 2).alias("margin"),
    )
    .orderBy(F.desc("revenue"))
    .limit(10)
)

same = top_groups_sql.exceptAll(top_groups_api).count() == 0 and top_groups_api.exceptAll(top_groups_sql).count() == 0
print("Результаты SQL и DataFrame API совпадают:", same)

Результаты SQL и DataFrame API совпадают: True


In [20]:
segments_sql = spark.sql('''
SELECT
    CASE WHEN buyer = 'Частное лицо' THEN 'Розничные покупатели'
         ELSE 'Юридические лица' END AS segment,
    COUNT(DISTINCT client_id) AS clients,
    COUNT(*) AS lines,
    CAST(ROUND(SUM(amount), 2) AS DECIMAL(18, 2)) AS revenue,
    ROUND(100 * SUM(amount) / (SELECT SUM(amount) FROM komus_sales), 2) AS revenue_share_pct,
    ROUND(AVG(CASE WHEN is_delivery THEN 1 ELSE 0 END) * 100, 1) AS delivery_pct
FROM komus_sales
GROUP BY segment
ORDER BY revenue DESC
''')

segments_sql.show(truncate=False)

+--------------------+-------+------+------------+-----------------+------------+
|segment             |clients|lines |revenue     |revenue_share_pct|delivery_pct|
+--------------------+-------+------+------------+-----------------+------------+
|Розничные покупатели|1      |827666|103198746.29|78.34            |0.0         |
|Юридические лица    |1931   |37556 |28539819.05 |21.66            |88.0        |
+--------------------+-------+------+------------+-----------------+------------+



In [21]:
margin_check_sql = spark.sql('''
SELECT
    COUNT(*) AS groups,
    ROUND(MIN(margin_pct), 2) AS min_margin_pct,
    ROUND(MAX(margin_pct), 2) AS max_margin_pct,
    ROUND(AVG(margin_pct), 2) AS avg_margin_pct
FROM (
    SELECT
        product_group,
        try_divide(100 * SUM(margin),
                   SUM(CASE WHEN cost IS NOT NULL THEN amount END)) AS margin_pct
    FROM komus_sales
    GROUP BY product_group
    HAVING COUNT(*) >= 500
) g
''')

margin_check_sql.show()

+------+--------------+--------------+--------------+
|groups|min_margin_pct|max_margin_pct|avg_margin_pct|
+------+--------------+--------------+--------------+
|   218|         48.44|          53.1|         50.02|
+------+--------------+--------------+--------------+



**Вывод.** Первый запрос показал, что наибольшую выручку дают холсты на подрамнике (4,41 млн), уценённые товары (3,30 млн) и акварельные наборы (3,18 млн); его результат полностью совпал с эквивалентной агрегацией через DataFrame API. Второй запрос разделил продажи на сегменты: обезличенный клиент «Частное лицо» даёт 78,34% выручки, а 1 931 юридическое лицо — 21,66%, причём продажи юрлицам в 88% строк сопровождаются доставкой. Третий запрос показал, что в 218 товарных группах с объёмом не менее 500 строк доля валовой маржи лежит в узком диапазоне от 48.44 до 53.1%, что указывает на расчётный характер себестоимости.

## 10. Итоговая витрина и запись в Parquet и CSV

In [22]:
group_mart = (
    mart
    .groupBy("product_group")
    .agg(
        F.countDistinct("product_key").alias("products"),
        F.count("*").alias("lines"),
        F.countDistinct("receipt_no").alias("receipts"),
        F.sum("quantity").alias("quantity"),
        F.round(F.sum("amount"), 2).cast("decimal(18,2)").alias("revenue"),
        F.round(F.sum("margin"), 2).cast("decimal(18,2)").alias("margin"),
        F.sum(F.when(F.col("cost").isNotNull(), F.col("amount"))).alias("revenue_with_cost"),
    )
    .withColumn(
        "margin_pct",
        F.when(
            F.col("revenue_with_cost") > 0,
            F.round(F.col("margin") / F.col("revenue_with_cost") * 100, 2),
        ),
    )
    .drop("revenue_with_cost")
)

print("Товарных групп в витрине:", group_mart.count())
group_mart.printSchema()

OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

group_mart.write.mode("overwrite").parquet(str(OUT / "komus_group_mart_parquet"))

(
    group_mart
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("escape", '"')
    .csv(str(OUT / "komus_group_mart_csv"))
)

Товарных групп в витрине: 437
root
 |-- product_group: string (nullable = true)
 |-- products: long (nullable = false)
 |-- lines: long (nullable = false)
 |-- receipts: long (nullable = false)
 |-- quantity: long (nullable = true)
 |-- revenue: decimal(18,2) (nullable = true)
 |-- margin: decimal(18,2) (nullable = true)
 |-- margin_pct: double (nullable = true)



## 11. Повторное чтение Parquet

In [23]:
group_back = spark.read.parquet(str(OUT / "komus_group_mart_parquet"))

print("Строк до записи:", group_mart.count())
print("Строк после чтения:", group_back.count())

same_schema = [(f.name, f.dataType) for f in group_mart.schema] == [(f.name, f.dataType) for f in group_back.schema]
print("Имена и типы столбцов совпадают:", same_schema)
group_back.printSchema()

Строк до записи: 437
Строк после чтения: 437
Имена и типы столбцов совпадают: True
root
 |-- product_group: string (nullable = true)
 |-- products: long (nullable = true)
 |-- lines: long (nullable = true)
 |-- receipts: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- revenue: decimal(18,2) (nullable = true)
 |-- margin: decimal(18,2) (nullable = true)
 |-- margin_pct: double (nullable = true)



**Вывод.** Итоговая витрина по 437 товарным группам записана в Parquet и CSV; для CSV использован `coalesce(1)`, допустимый при таком малом объёме. Повторное чтение Parquet вернуло те же 437 строк, имена и типы столбцов совпали; различается только признак `nullable`, который Parquet сохраняет как `true` для всех столбцов.

## 12. Итоговые выводы и ограничения

**Итоговые выводы**

1. Данные КОМУС за период с 16.06.2020 по 31.12.2021 (865 222 строки продаж) обработаны в Spark без staging-этапа: формат CSV читается напрямую при явном указании разделителя, кодировки, экранирования кавычек и обозначения пропусков.
2. Явное описание формата важнее автоматического вывода типов: `inferSchema` не распознал числа с десятичной запятой, а объявление их числовым типом привело бы к потере 196 690 значений без ошибки.
3. Пропуски в наборе преимущественно структурные: отсутствие машины доставки означает продажу без доставки, отсутствие номера чека — продажу без кассового чека; удаление таких строк исказило бы выручку.
4. Выручка сконцентрирована в рознице: обезличенные частные покупатели дают 78,3% выручки, а крупнейший филиал ТЗ-2 — около 45% (58,75 из 131,74 млн).
5. Рост выручки во второй половине 2021 года объясняется подключением новых торговых точек, а не ростом продаж в одном филиале, поэтому динамику корректнее сравнивать по филиалам отдельно.
6. Контроль числа строк после JOIN выявил дубликаты в справочнике клиентов: без их устранения в витрину попали бы 9 351 лишняя строка, и выручка была бы завышена.

**Ограничения анализа**

1. Доля валовой маржи почти одинакова во всех филиалах (около 50%) и в крупных товарных группах (от 48 до 53%), поэтому себестоимость, по-видимому, рассчитана по нормативу, а не взята из учёта; кроме того, для 10% выручки она неизвестна. Выводы о прибыльности товаров на её основе ненадёжны.
2. Из данных не ясно, учтена ли скидка в сумме покупки, поэтому показатели выручки могут отличаться от фактически оплаченных сумм; таблица «Суммы чеков» для сверки не использовалась.
3. Анализ выполнен в локальном режиме на одном компьютере: он подтверждает корректность программной модели Spark, но не характеризует производительность на кластере.

In [24]:
spark.stop()